Comparing the tables

In [1]:
import pyspark
import pandas as pd
import numpy as np


from pyspark.sql import SparkSession
from pyspark.sql.functions import countDistinct
spark = SparkSession.builder.appName('practice').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/07/20 23:21:17 WARN Utils: Your hostname, Janna-MBP.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.26 instead (on interface en0)
25/07/20 23:21:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/20 23:21:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_pd = pd.read_csv("week2-ds.tsv", sep = '\t')
df_pd.head()

,Polygon ID,Hashed Device ID,Unix Timestamp of Visit,Data Source,Date,Time of Day,Day of Week,Time Zone
0,"Blue Nile Blue Nile Jewelry - Fashion Square, ...",9f0bc33ace6a4d2bdd678ae1a64f822646e5fadf,1643151728,X,1/25/22,16:02:08,Tue,America/Phoenix
1,"Blue Nile Blue Nile Jewelry - Fashion Square, ...",ca58bf6ffe8a0cef2d91d39d24eb030228cb4ab1,1643131230,X,1/25/22,10:20:30,Tue,America/Phoenix
2,"Blue Nile Blue Nile Jewelry - Fashion Square, ...",0c275358edf2e911bbfcfab93305f487aa42121f,1643153377,X,1/25/22,16:29:37,Tue,America/Phoenix
3,"Blue Nile Blue Nile Jewelry - Fashion Square, ...",30a03daca53ba0f547196a63e3964dd0fde3d66d,1643142816,X,1/25/22,13:33:36,Tue,America/Phoenix
4,"Blue Nile Blue Nile Jewelry - Fashion Square, ...",b7c057bbd95d16c97b84ac52fb3b8c9691e78f99,1643125135,X,1/25/22,8:38:55,Tue,America/Phoenix


In [3]:
df_ps = spark.read.option('header', 'true').csv('week2-ds.tsv', sep = '\t')
df_ps.show(5)

+--------------------+--------------------+-----------------------+-----------+-------+-----------+-----------+---------------+
|          Polygon ID|    Hashed Device ID|Unix Timestamp of Visit|Data Source|   Date|Time of Day|Day of Week|      Time Zone|
+--------------------+--------------------+-----------------------+-----------+-------+-----------+-----------+---------------+
|Blue Nile Blue Ni...|9f0bc33ace6a4d2bd...|             1643151728|          X|1/25/22|   16:02:08|        Tue|America/Phoenix|
|Blue Nile Blue Ni...|ca58bf6ffe8a0cef2...|             1643131230|          X|1/25/22|   10:20:30|        Tue|America/Phoenix|
|Blue Nile Blue Ni...|0c275358edf2e911b...|             1643153377|          X|1/25/22|   16:29:37|        Tue|America/Phoenix|
|Blue Nile Blue Ni...|30a03daca53ba0f54...|             1643142816|          X|1/25/22|   13:33:36|        Tue|America/Phoenix|
|Blue Nile Blue Ni...|b7c057bbd95d16c97...|             1643125135|          X|1/25/22|    8:38:55|     

In [4]:
df1_pd = df_pd.groupby('Date')['Hashed Device ID'].nunique().reset_index(name='Hashed Device ID')
df1_pd

,Date,Hashed Device ID
0,1/1/22,78
1,1/10/22,41
2,1/11/22,44
3,1/12/22,44
4,1/13/22,47
...,...,...
177,6/5/22,35
178,6/6/22,39
179,6/7/22,32
180,6/8/22,27


In [5]:
df1_ps = df_ps.groupBy('Date').agg(countDistinct('Hashed Device ID')).orderBy("Date")
df1_ps.show()

+-------+--------------------------------+
|   Date|count(DISTINCT Hashed Device ID)|
+-------+--------------------------------+
| 1/1/22|                              78|
|1/10/22|                              41|
|1/11/22|                              44|
|1/12/22|                              44|
|1/13/22|                              47|
|1/14/22|                              64|
|1/15/22|                             121|
|1/16/22|                              62|
|1/17/22|                              71|
|1/18/22|                              35|
|1/19/22|                              31|
| 1/2/22|                              61|
|1/20/22|                              42|
|1/21/22|                              48|
|1/22/22|                              99|
|1/23/22|                              54|
|1/24/22|                              31|
|1/25/22|                              29|
|1/26/22|                              39|
|1/27/22|                              36|
+-------+--

In [6]:
df1_ps_pd = df1_ps.toPandas()

comparison_result = np.array_equal(df1_pd.values, df1_ps_pd.values)

print(f"The data frames are equal: {comparison_result}")


The data frames are equal: True


In [7]:
df2_pd = df_pd.groupby('Polygon ID')['Hashed Device ID'].nunique().reset_index(name='Hashed Device ID')
df2_pd

,Polygon ID,Hashed Device ID
0,Blue Nile Blue Nile Jewelry - Domain Northside...,396
1,"Blue Nile Blue Nile Jewelry - Fashion Square, ...",2884
2,Blue Nile Blue Nile Jewelry - Garden State Pla...,726
3,Blue Nile Blue Nile Jewelry - Houston Galleria...,1962
4,Blue Nile Blue Nile Jewelry - Roseville Galler...,963
5,"Blue Nile The Mall at Rockingham Park, Salem, ...",426
6,"Blue Nile Washington Square, Portland, OR|9695391",164


In [8]:
df2_ps = df_ps.groupBy('Polygon ID').agg(countDistinct('Hashed Device ID')).orderBy("Polygon ID")
df2_ps.show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|Polygon ID                                                                                                                   |count(DISTINCT Hashed Device ID)|
+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622                                                          |396                             |
|Blue Nile Blue Nile Jewelry - Fashion Square, Scottsdale, AZ|11088822                                                        |2884                            |
|Blue Nile Blue Nile Jewelry - Garden State Plaza, Paramus, NJ|11050629                                                       |726                             |
|Blue Nile Blue Nile Jewelry - Hou

In [9]:
df2_ps_pd = df2_ps.toPandas()

comparison_result = np.array_equal(df2_pd.values, df2_ps_pd.values)

print(f"The data frames are equal: {comparison_result}")

The data frames are equal: True


In [10]:
df3_pd = df_pd.groupby(['Polygon ID', 'Date'])['Hashed Device ID'].nunique().reset_index(name='Hashed Device ID')
df3_pd

,Polygon ID,Date,Hashed Device ID
0,Blue Nile Blue Nile Jewelry - Domain Northside...,1/1/22,10
1,Blue Nile Blue Nile Jewelry - Domain Northside...,1/10/22,8
2,Blue Nile Blue Nile Jewelry - Domain Northside...,1/11/22,9
3,Blue Nile Blue Nile Jewelry - Domain Northside...,1/12/22,5
4,Blue Nile Blue Nile Jewelry - Domain Northside...,1/13/22,7
...,...,...,...
1210,"Blue Nile Washington Square, Portland, OR|9695391",6/5/22,2
1211,"Blue Nile Washington Square, Portland, OR|9695391",6/6/22,3
1212,"Blue Nile Washington Square, Portland, OR|9695391",6/7/22,2
1213,"Blue Nile Washington Square, Portland, OR|9695391",6/8/22,1


In [11]:
df3_ps = df_ps.groupBy(['Polygon ID', 'Date']).agg(countDistinct('Hashed Device ID')).orderBy("Polygon ID", "Date")
df3_ps.show(truncate=False)

+-------------------------------------------------------------------+-------+--------------------------------+
|Polygon ID                                                         |Date   |count(DISTINCT Hashed Device ID)|
+-------------------------------------------------------------------+-------+--------------------------------+
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/1/22 |10                              |
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/10/22|8                               |
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/11/22|9                               |
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/12/22|5                               |
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/13/22|7                               |
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/14/22|7                               |
|

In [12]:
df3_ps_pd = df3_ps.toPandas()

comparison_result = np.array_equal(df3_pd.values, df3_ps_pd.values)

print(f"The data frames are equal: {comparison_result}")

The data frames are equal: True


25/07/21 03:32:12 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 441649 ms exceeds timeout 120000 ms
25/07/21 03:32:12 WARN SparkContext: Killing executors is not supported by current scheduler.
25/07/21 03:32:16 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o